In [2]:
# =============================================================================
# GRADIENT BOOSTING + OPTUNA - MAXIMIZE GENERALIZATION
# Based on your 0.798 GB model, now with intelligent tuning
# Expected: CV ~0.88-0.90, Kaggle 0.81-0.83
# Runtime: ~2-3 hours (GB is slower than XGBoost)   op dataset6 tips guilem 29/11 14:15
# =============================================================================

In [3]:
import numpy as np
import pandas as pd
import optuna
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import roc_auc_score
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("GRADIENT BOOSTING + OPTUNA - CONSERVATIVE TUNING")
print("="*80)

GRADIENT BOOSTING + OPTUNA - CONSERVATIVE TUNING


In [4]:


# =============================================================================
# 1. LOAD DATA
# =============================================================================

# =============================================================================
# 1. LOAD PREPROCESSED DATA (no need to preprocess again!)
# =============================================================================
print("Loading preprocessed data...")
X = pd.read_pickle('../../data/processed/X_train_processed.pkl')
y = pd.read_pickle('../../data/processed/y_train.pkl')
X_test = pd.read_pickle('../../data/processed/X_test_processed.pkl')
test_ids = pd.read_pickle('../../data/processed/test_ids.pkl')

print(f"X_train: {X.shape}")
print(f"y_train: {y.shape}")
print(f"Class imbalance: {y.mean():.3f}")

# =============================================================================
# 2. OPTUNA OBJECTIVE FUNCTION - GRADIENT BOOSTING
# =============================================================================

def objective(trial):
    """
    Gradient Boosting parameter search
    Focus: Maximum generalization (minimize CV-Kaggle gap)
    """
    
    params = {
        # Number of trees
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        
        # Learning rate - LOWER for better generalization
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        
        # Tree depth - SHALLOW for generalization
        'max_depth': trial.suggest_int('max_depth', 2, 6),
        
        # Minimum samples - HIGH values for regularization
        'min_samples_split': trial.suggest_int('min_samples_split', 10, 100),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 5, 50),
        
        # Sampling - helps prevent overfitting
        'subsample': trial.suggest_float('subsample', 0.5, 0.95),
        
        # Feature sampling
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.5, 0.7, 0.9]),
        
        # Fixed
        'random_state': 42,
        'verbose': 0
    }
    
    # 5-fold CV
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    model = GradientBoostingClassifier(**params)
    
    # Cross-validation
    cv_scores = cross_val_score(
        model, X, y,
        cv=cv,
        scoring='roc_auc',
        n_jobs=-1
    )
    
    cv_mean = cv_scores.mean()
    cv_std = cv_scores.std()
    
    # Penalize high variance across folds
    # This encourages models that generalize well
    score = cv_mean - 0.5 * cv_std
    
    return score

# =============================================================================
# 3. RUN OPTUNA OPTIMIZATION
# =============================================================================

print("\n" + "="*80)
print("STARTING OPTUNA OPTIMIZATION")
print("="*80)
print("\nSettings:")
print("  Trials: 100")
print("  CV folds: 5")
print("  Algorithm: Gradient Boosting (sklearn)")
print("  Focus: Maximize generalization")
print("  Expected runtime: ~2-3 hours")
print("\nOptimization progress:")

# Callback to show progress
def callback(study, trial):
    if trial.number % 10 == 0:
        print(f"  Trial {trial.number:3d}: Score={trial.value:.4f} (Best so far: {study.best_value:.4f})")

# Create study
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42)
)

# Optimize
study.optimize(
    objective,
    n_trials=100,
    callbacks=[callback],
    show_progress_bar=True,
    n_jobs=1  # GB doesn't parallelize well
)

# =============================================================================
# 4. RESULTS
# =============================================================================

print("\n" + "="*80)
print("OPTUNA RESULTS")
print("="*80)

print(f"\nBest trial:")
print(f"  Score (CV - 0.5*std): {study.best_value:.4f}")

print(f"\nBest parameters:")
best_params = study.best_params
for param, value in best_params.items():
    print(f"  {param}: {value}")

# =============================================================================
# 5. TRAIN FINAL MODEL WITH BEST PARAMS
# =============================================================================

print("\n" + "="*80)
print("TRAINING FINAL MODEL")
print("="*80)

# Add fixed params
final_params = best_params.copy()
final_params.update({
    'random_state': 42,
    'verbose': 0
})

# Train with best params
final_model = GradientBoostingClassifier(**final_params)

# Get CV score with best params
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(final_model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)

print(f"\nFinal CV Performance:")
print(f"  CV scores: {[f'{s:.4f}' for s in cv_scores]}")
print(f"  Mean: {cv_scores.mean():.4f}")
print(f"  Std: {cv_scores.std():.4f}")

# Train on full data
print("\nTraining on full dataset...")
final_model.fit(X, y)

# Training performance
train_proba = final_model.predict_proba(X)[:, 1]
train_auc = roc_auc_score(y, train_proba)

print(f"\nOverfitting Analysis:")
print(f"  Training AUC: {train_auc:.4f}")
print(f"  CV AUC: {cv_scores.mean():.4f}")
print(f"  Gap: {train_auc - cv_scores.mean():.4f}")

gap = train_auc - cv_scores.mean()
if gap < 0.05:
    print("  ✓ Excellent! Minimal overfitting")
elif gap < 0.10:
    print("  ✓ Good! Acceptable overfitting")
elif gap < 0.15:
    print("  ⚠️ Moderate overfitting")
else:
    print("  ❌ High overfitting")

# =============================================================================
# 6. GENERATE PREDICTIONS
# =============================================================================

print("\n--- Generating test predictions ---")

test_proba = final_model.predict_proba(X_test)[:, 1]

print(f"\nTest prediction statistics:")
print(f"  Min: {test_proba.min():.4f}")
print(f"  Max: {test_proba.max():.4f}")
print(f"  Mean: {test_proba.mean():.4f}")
print(f"  Median: {np.median(test_proba):.4f}")

# Distribution
print(f"\nPrediction distribution:")
bins = [0, 0.05, 0.10, 0.15, 0.20, 0.30, 1.0]
for i in range(len(bins)-1):
    count = ((test_proba >= bins[i]) & (test_proba < bins[i+1])).sum()
    print(f"  {bins[i]:.2f}-{bins[i+1]:.2f}: {count:4d} ({count/len(test_proba)*100:5.1f}%)")

[I 2025-11-29 14:18:39,322] A new study created in memory with name: no-name-11c2fab7-7d94-4644-9b4b-fe25d06306ca


Loading preprocessed data...
X_train: (20885, 95)
y_train: (20885,)
Class imbalance: 0.112

STARTING OPTUNA OPTIMIZATION

Settings:
  Trials: 100
  CV folds: 5
  Algorithm: Gradient Boosting (sklearn)
  Focus: Maximize generalization
  Expected runtime: ~2-3 hours

Optimization progress:


Best trial: 0. Best value: 0.89595:   1%|          | 1/100 [00:04<07:42,  4.67s/it]

[I 2025-11-29 14:18:43,990] Trial 0 finished with value: 0.8959499849765815 and parameters: {'n_estimators': 250, 'learning_rate': 0.13125830316209655, 'max_depth': 5, 'min_samples_split': 64, 'min_samples_leaf': 12, 'subsample': 0.5701975341512912, 'max_features': 'log2'}. Best is trial 0 with value: 0.8959499849765815.
  Trial   0: Score=0.8959 (Best so far: 0.8959)


Best trial: 1. Best value: 0.898193:   2%|▏         | 2/100 [00:28<26:30, 16.23s/it]

...

[I 2025-11-29 16:36:03,415] Trial 97 finished with value: 0.9020878438841731 and parameters: {'n_estimators': 473, 'learning_rate': 0.03350958376232216, 'max_depth': 6, 'min_samples_split': 73, 'min_samples_leaf': 11, 'subsample': 0.9391617615397743, 'max_features': 0.9}. Best is trial 35 with value: 0.9037033643031896.


Best trial: 35. Best value: 0.903703:  99%|█████████▉| 99/100 [2:19:10<03:11, 191.70s/it]

[I 2025-11-29 16:37:49,485] Trial 98 finished with value: 0.8974945629580335 and parameters: {'n_estimators': 207, 'learning_rate': 0.03042776786212686, 'max_depth': 6, 'min_samples_split': 76, 'min_samples_leaf': 12, 'subsample': 0.9080815282245482, 'max_features': 0.9}. Best is trial 35 with value: 0.9037033643031896.


Best trial: 35. Best value: 0.903703: 100%|██████████| 100/100 [2:19:39<00:00, 83.79s/it] 


[I 2025-11-29 16:38:18,662] Trial 99 finished with value: 0.9003564109633029 and parameters: {'n_estimators': 492, 'learning_rate': 0.02664476820121502, 'max_depth': 6, 'min_samples_split': 69, 'min_samples_leaf': 6, 'subsample': 0.661596921360641, 'max_features': 'sqrt'}. Best is trial 35 with value: 0.9037033643031896.

OPTUNA RESULTS

Best trial:
  Score (CV - 0.5*std): 0.9037

Best parameters:
  n_estimators: 497
  learning_rate: 0.0300400842308156
  max_depth: 6
  min_samples_split: 78
  min_samples_leaf: 11
  subsample: 0.8946113169690653
  max_features: 0.9

TRAINING FINAL MODEL

Final CV Performance:
  CV scores: ['0.9016', '0.9057', '0.9100', '0.9212', '0.8993']
  Mean: 0.9076
  Std: 0.0077

Training on full dataset...

Overfitting Analysis:
  Training AUC: 0.9790
  CV AUC: 0.9076
  Gap: 0.0714
  ✓ Good! Acceptable overfitting

--- Generating test predictions ---

Test prediction statistics:
  Min: 0.0010
  Max: 0.9954
  Mean: 0.1130
  Median: 0.0296

Prediction distribution:


In [5]:
# =============================================================================
# 7. SAVE SUBMISSION
# =============================================================================

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
submission_file = f"../../outputs/predictions/gb_optuna_{timestamp}.csv"

submission = pd.DataFrame({
    'icustay_id': test_ids,
    'prediction': test_proba
})

submission.to_csv(submission_file, index=False)
print(f"\n✓ Saved: {submission_file}")


✓ Saved: ../../outputs/predictions/gb_optuna_20251129_1645.csv


In [6]:
# =============================================================================
# 8. FEATURE IMPORTANCE
# =============================================================================

print("\n" + "="*80)
print("TOP 20 FEATURES")
print("="*80)

feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\n" + feature_importance.head(20).to_string(index=False))

# =============================================================================
# 10. SAVE MODEL & STUDY
# =============================================================================

import pickle

# Save model
model_file = f"../../outputs/models/gb_optuna_{timestamp}.pkl"
with open(model_file, 'wb') as f:
    pickle.dump(final_model, f)
print(f"\n✓ Model saved: {model_file}")

# Save optuna study
study_file = f"../../outputs/models/gb_study_{timestamp}.pkl"
with open(study_file, 'wb') as f:
    pickle.dump(study, f)
print(f"✓ Study saved: {study_file}")




TOP 20 FEATURES

                feature  importance
   primary_diag_encoded    0.110902
           ICD9_encoded    0.084471
             TempC_Mean    0.057076
              SysBP_Min    0.056747
               SpO2_Min    0.055046
              SpO2_Mean    0.052261
has_respiratory_failure    0.048092
            n_diagnoses    0.031495
             has_sepsis    0.031051
            Glucose_Min    0.023779
             SysBP_Mean    0.023356
          RespRate_Mean    0.022946
             Temp_Range    0.022374
                has_aki    0.021461
           Glucose_Mean    0.020858
              TempC_Max    0.020408
             ShockIndex    0.018350
     ModifiedShockIndex    0.018328
              TempC_Min    0.017566
          HeartRate_Min    0.016212

✓ Model saved: ../../outputs/models/gb_optuna_20251129_1645.pkl
✓ Study saved: ../../outputs/models/gb_study_20251129_1645.pkl


In [7]:
# =============================================================================
# 11. OPTUNA VISUALIZATION (OPTIONAL)
# =============================================================================

print(f"\n{'='*80}")
print("OPTUNA STUDY INSIGHTS")
print(f"{'='*80}")

# Best trials
print(f"\nTop 5 trials:")
best_trials = sorted(study.trials, key=lambda t: t.value, reverse=True)[:5]
for i, trial in enumerate(best_trials, 1):
    print(f"\n  {i}. Trial {trial.number}: Score={trial.value:.4f}")
    print(f"     n_estimators={trial.params['n_estimators']}, " +
          f"lr={trial.params['learning_rate']:.4f}, " +
          f"depth={trial.params['max_depth']}")

# Parameter importance (if enough trials)
if len(study.trials) >= 20:
    print(f"\nMost important parameters (based on Optuna):")
    try:
        importance = optuna.importance.get_param_importances(study)
        for param, imp in sorted(importance.items(), key=lambda x: x[1], reverse=True)[:5]:
            print(f"  {param}: {imp:.3f}")
    except:
        print("  (Could not compute - needs more trials)")

print("\n" + "="*80)
print("OPTIMIZATION COMPLETE!")
print("="*80)
print("\n🎯 Key takeaway: GB's conservative nature might finally beat XGBoost!")
print("   Your original GB got 0.798 with random search (30 trials)")
print("   This optimized version should push to 0.81-0.83! 🚀")


OPTUNA STUDY INSIGHTS

Top 5 trials:

  1. Trial 35: Score=0.9037
     n_estimators=497, lr=0.0300, depth=6

  2. Trial 39: Score=0.9033
     n_estimators=500, lr=0.0364, depth=6

  3. Trial 83: Score=0.9033
     n_estimators=479, lr=0.0285, depth=6

  4. Trial 93: Score=0.9033
     n_estimators=472, lr=0.0287, depth=6

  5. Trial 34: Score=0.9033
     n_estimators=456, lr=0.0316, depth=6

Most important parameters (based on Optuna):
  n_estimators: 0.468
  max_depth: 0.443
  min_samples_leaf: 0.027
  learning_rate: 0.025
  max_features: 0.017

OPTIMIZATION COMPLETE!

🎯 Key takeaway: GB's conservative nature might finally beat XGBoost!
   Your original GB got 0.798 with random search (30 trials)
   This optimized version should push to 0.81-0.83! 🚀
